In [1]:
from pathlib import Path
import sys
import torch 

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0,str(PROJECT_ROOT))

from instancepipelineprior import get_data_loaders
from bayesianprior.discretizedtrain import EarthquakeCNN, evaluate_metrics, logits_to_magnitude

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint_path = PROJECT_ROOT / "bayesianprior" / "data" / "best_model_huberandcross5s075.pth"

checkpoint= torch.load(checkpoint_path, map_location=device)
model = EarthquakeCNN().to(device)

model.load_state_dict(checkpoint["model_state_dict"])

model.eval()




Train samples: 979487
Validation samples: 96993
Test samples: 82769
Train batches: 7653
Val batches: 758
Test batches: 647
Train samples: 979487
Validation samples: 96993
Test samples: 82769
Train batches: 7653
Val batches: 758
Test batches: 647


EarthquakeCNN(
  (features): Sequential(
    (0): Sequential(
      (0): Conv1d(3, 16, kernel_size=(5,), stride=(1,), padding=(2,))
      (1): ReLU()
      (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (1): Sequential(
      (0): Conv1d(16, 32, kernel_size=(5,), stride=(1,), padding=(2,))
      (1): ReLU()
      (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (2): Sequential(
      (0): Conv1d(32, 64, kernel_size=(5,), stride=(1,), padding=(2,))
      (1): ReLU()
      (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (3): Sequential(
      (0): Conv1d(64, 128, kernel_size=(5,), stride=(1,), padding=(2,))
      (1): ReLU()
      (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (4): Sequential(
      (0): Conv1d(128, 256, kernel_size=(5,), stride=(1,), padding=(2,))
      (1): ReLU()
      (2): MaxPool1d(kernel_size=2, str

In [2]:
train_loader,val_loader,test_loader = get_data_loaders()


In [3]:
import torch

smoothed_prior = torch.load(
    "data/smoothed_magnitude_prior.pt",
    weights_only=False
)

In [4]:
smoothed_prior = torch.tensor(
    smoothed_prior,
    dtype=torch.float32
)

In [5]:
def logits_to_magnitude_with_prior(logits,bin_centers,prior_probs,alpha):
    prior_probs = prior_probs.to(logits.device)
    bin_centers = bin_centers.to(logits.device)
    log_prior = torch.log(prior_probs.clamp(min=1e-8))

    adjusted_logits = (logits + (alpha - 1.0) * log_prior.unsqueeze(0))
    adjusted_probs = torch.softmax(adjusted_logits,dim=1)
    predictions = torch.sum(adjusted_probs* bin_centers.unsqueeze(0),dim=1)
    return predictions, adjusted_probs

In [6]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

In [7]:
waveforms, magnitudes = next(
    iter(val_loader)
)

waveforms = waveforms.to(device)
magnitudes = magnitudes.to(device)

print(waveforms.shape)
print(magnitudes.shape)

torch.Size([128, 3, 300])
torch.Size([128])


In [8]:
MIN_MAGNITUDE = 0.0
MAX_MAGNITUDE = 6.6
BIN_WIDTH = 0.1

BIN_CENTERS = torch.arange(
    MIN_MAGNITUDE + BIN_WIDTH / 2,
    MAX_MAGNITUDE,
    BIN_WIDTH,
    dtype=torch.float32
)

In [9]:
logits = model(waveforms)
predictions, probs = logits_to_magnitude_with_prior(logits,BIN_CENTERS,smoothed_prior,alpha=1.0)

In [10]:
original_predictions = logits_to_magnitude(
    logits,
    BIN_CENTERS.to(device)
)

print(
    torch.max(
        torch.abs(
            original_predictions - predictions
        )
    )
)

print(
    torch.allclose(
        original_predictions,
        predictions,
        atol=1e-6
    )
)

tensor(0., device='cuda:0', grad_fn=<MaxBackward1>)
True


In [11]:
import pandas as pd

In [12]:
def evaluate_metrics_with_prior(model, dataloader, device, prior_probs, alpha):
    model.eval()

    all_predictions = []
    all_targets = []
    bin_centers = BIN_CENTERS.to(device)

    with torch.no_grad():
        for waveforms, magnitudes in dataloader:
            waveforms = waveforms.to(device, non_blocking=True)
            magnitudes = magnitudes.to(device, non_blocking=True)

            logits = model(waveforms)

            predictions, _ = logits_to_magnitude_with_prior(
                logits,
                bin_centers,
                prior_probs,
                alpha
            )

            all_predictions.append(predictions.cpu())
            all_targets.append(magnitudes.cpu())

    all_predictions = torch.cat(all_predictions, dim=0)
    all_targets = torch.cat(all_targets, dim=0)

    errors = all_predictions - all_targets

    mae = torch.abs(errors).mean().item()
    rmse = torch.sqrt(torch.mean(errors ** 2)).item()
    bias = errors.mean().item()
    accuracy = (torch.abs(errors) <= 0.2).float().mean().item() * 100

    results = pd.DataFrame({
        "true_magnitude": all_targets.numpy(),
        "predicted_magnitude": all_predictions.numpy()
    })

    return results, mae, rmse, bias, accuracy


In [13]:
alphas = [
    0.0,
    0.25,
    0.5,
    0.75,
    1.0,
    1.25
]

In [14]:
import numpy as np

In [15]:

maelist = []
rmselist = []
biaslist = []
accuracylist = []
bin_rows = []

for alpha in alphas:
    results, mae, rmse, bias, accuracy = evaluate_metrics_with_prior(model, val_loader, device, smoothed_prior, alpha)

    maelist.append(mae)
    rmselist.append(rmse)
    biaslist.append(bias)
    accuracylist.append(accuracy)
    results.to_csv(f"5secondsalpha{alpha}.csv")
    results["error"] = results["predicted_magnitude"] - results["true_magnitude"]
    results["squared_error"] = results["error"] ** 2

    results["bin"] = pd.cut(
        results["true_magnitude"],
        bins=[0, 1, 2, 3, 4, 5, 6],
        labels=["0-1", "1-2", "2-3", "3-4", "4-5", "5-6"],
        include_lowest=True
    )

    bin_results = results.groupby("bin", observed=True).agg(
        bias=("error", "mean"),
        mse=("squared_error", "mean")
    )

    bin_results["rmse"] = np.sqrt(bin_results["mse"])

    row = {"alpha": alpha}

    for bin_name in ["0-1", "1-2", "2-3", "3-4", "4-5", "5-6"]:
        row[f"{bin_name} Bias"] = bin_results.loc[bin_name, "bias"] if bin_name in bin_results.index else np.nan
        row[f"{bin_name} RMSE"] = bin_results.loc[bin_name, "rmse"] if bin_name in bin_results.index else np.nan

    bin_rows.append(row)

bin_df = pd.DataFrame(bin_rows)

columns = ["alpha"] + [f"{b} Bias" for b in ["0-1", "1-2", "2-3", "3-4", "4-5", "5-6"]] + [f"{b} RMSE" for b in ["0-1", "1-2", "2-3", "3-4", "4-5", "5-6"]]
bin_df = bin_df[columns]

overall_df = pd.DataFrame({
    "alpha": alphas,
    "MAE": maelist,
    "RMSE": rmselist,
    "Bias": biaslist,
    "Accuracy ±0.2": accuracylist
})

display(overall_df)
display(bin_df)



,alpha,MAE,RMSE,Bias,Accuracy ±0.2
0,0.00,0.556490,0.775527,0.134771,27.511263
1,0.25,0.497302,0.695155,0.125028,30.327961
2,0.50,0.458146,0.636543,0.115941,32.319859
3,0.75,0.433468,0.596580,0.109105,33.760169
4,1.00,0.418997,0.571194,0.104530,34.592187
5,1.25,0.412267,0.557077,0.101311,35.171610


,alpha,0-1 Bias,1-2 Bias,2-3 Bias,3-4 Bias,4-5 Bias,5-6 Bias,0-1 RMSE,1-2 RMSE,2-3 RMSE,3-4 RMSE,4-5 RMSE,5-6 RMSE
0,0.00,0.616722,0.203975,0.043488,0.016855,-0.195995,-0.578260,1.099165,0.742573,0.728044,0.824036,0.762124,1.132289
1,0.25,0.699169,0.244411,0.010375,-0.123589,-0.342862,-0.730425,1.052613,0.667353,0.628608,0.764305,0.815553,1.246261
2,0.50,0.762048,0.278779,-0.016935,-0.250041,-0.488213,-0.879034,1.027205,0.614208,0.547561,0.732269,0.885498,1.351963
3,0.75,0.813495,0.310191,-0.038616,-0.361582,-0.631524,-1.031602,1.019274,0.579538,0.482286,0.723122,0.964475,1.447247
4,1.00,0.860247,0.339785,-0.056154,-0.459157,-0.773959,-1.209404,1.025291,0.558667,0.429205,0.730840,1.047847,1.536477
5,1.25,0.905208,0.367245,-0.071248,-0.544291,-0.916183,-1.437344,1.041212,0.547449,0.386005,0.750352,1.138576,1.659495
